# 01 · Harmonize and profile the real cohorts
Runs `harmonize.py`, checks the numbers quoted in the protocol, and plots the clinical priors against the data.
No model is trained here.

In [ ]:
import os, sys, subprocess, pathlib, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
sys.path.insert(0, ".")
import ckinn_lib as L
RICH_XLSX = os.environ.get("RICH_XLSX", "771969dd-RCHealthCareData-20180820.xlsx")
NAGALA_XLSX = os.environ.get("NAGALA_XLSX", "5a535e36-DatasetIntJObesity.xlsx")
DATA = pathlib.Path("data"); FIG = pathlib.Path("figs"); FIG.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

## 1. Build harmonized files

In [ ]:
subprocess.run([sys.executable, "harmonize.py", RICH_XLSX, NAGALA_XLSX, str(DATA)], check=True, capture_output=True)
print(sorted(p.name for p in DATA.iterdir()))

## 2. Check against the protocol numbers

In [ ]:
rich, nag = L.load_data(DATA)
assert len(rich) == 211833 and rich.event.sum() == 4174 and rich.event_dx_only.sum() == 1304
assert len(nag) == 15464 and nag.event.sum() == 373
print("Protocol numbers reproduced.")
print(open(DATA / "cohort_profile.md").read()[:1800])

## 3. Missingness (RICH) and site structure

In [ ]:
cols = ["tc", "tg", "alt", "hdl", "ast", "smoke"]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
rich[cols].isna().mean().mul(100).plot.bar(ax=ax[0], color="#4C78A8"); ax[0].set_ylabel("% missing"); ax[0].set_title("RICH missingness")
site = rich.groupby("site").agg(n=("id", "size"), smoke=("smoke", lambda s: s.isna().mean() * 100), hdl=("hdl", lambda s: s.isna().mean() * 100))
site[["smoke", "hdl"]].plot.bar(ax=ax[1]); ax[1].set_ylabel("% missing"); ax[1].set_title("By site")
plt.tight_layout(); plt.savefig(FIG / "01_missingness.png"); plt.show()
print("Event rate when HDL recorded vs missing:", rich.groupby(rich.hdl.isna()).event.mean().round(4).to_dict())

## 4. Do the clinical priors hold in the data? (event rate by quintile)

In [ ]:
vars_ = ["age", "bmi", "sbp", "tg", "fpg", "hdl"]
fig, axes = plt.subplots(2, 6, figsize=(17, 5.2), sharey="row")
for r_, (name, d) in enumerate([("RICH", rich), ("NAGALA", nag)]):
    for c_, v in enumerate(vars_):
        q = pd.qcut(d[v], 5, duplicates="drop")
        rate = d.groupby(q, observed=True).event.mean() * 100
        axes[r_, c_].plot(range(len(rate)), rate.values, "o-", color="#E45756" if L.PRIORS.get(v, 0) < 0 else "#4C78A8")
        axes[r_, c_].set_title(f"{name}: {v} ({'+' if L.PRIORS.get(v, 0) > 0 else '-'})", fontsize=9)
        axes[r_, c_].set_xlabel("quintile")
axes[0, 0].set_ylabel("event rate, %"); axes[1, 0].set_ylabel("event rate, %")
plt.tight_layout(); plt.savefig(FIG / "01_priors_vs_data.png"); plt.show()

## 5. Kaplan-Meier by cohort (shape of the censoring problem)

In [ ]:
from lifelines import KaplanMeierFitter
fig, ax = plt.subplots(figsize=(6, 3.8))
for name, d in [("RICH", rich), ("NAGALA", nag)]:
    KaplanMeierFitter().fit(d.time_y, d.event, label=name).plot_cumulative_density(ax=ax)
ax.axvline(L.H, ls="--", c="grey"); ax.set_xlabel("years"); ax.set_ylabel("cumulative incidence")
plt.tight_layout(); plt.savefig(FIG / "01_km.png"); plt.show()
for h in (2, 3, 5):
    print(f"RICH censored before {h} y: {((rich.event == 0) & (rich.time_y < h)).mean():.1%} | NAGALA: {((nag.event == 0) & (nag.time_y < h)).mean():.1%}")

## 6. Predictor tiers: complete cases

In [ ]:
rows = []
for t, cols in L.TIERS.items():
    cr, cn = rich.dropna(subset=cols), nag.dropna(subset=cols)
    rows.append({"tier": t, "variables": ", ".join(cols), "RICH n": len(cr), "RICH events": int(cr.event.sum()), "NAGALA n": len(cn), "NAGALA events": int(cn.event.sum())})
pd.DataFrame(rows)